In [1]:
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_wine, load_iris, fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score
from lime_ndt.lime_tabular import LimeNdtExplainer
from lime_ndt.utils.ndt_sklearn_wrapper import NDTRegressorWrapper
import lime.lime_tabular as lime_classic
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

ModuleNotFoundError: No module named 'lime_ndt'

In [ ]:
# LIME-NDT Fidelity & Stability Comparison with Other LIME Variants

This notebook compares the **fidelity** and **stability** of LIME-NDT with other LIME variants on the Wine dataset.

## Tested Variants in This Analysis

| Variant | Surrogate Model | Key Improvement | Limitations |
|---------|-----------------|-----------------|------------|
| **LIME-NDT** (ours) | Neural Decision Tree | Uses NDTs as surrogate models for higher fidelity | More computationally intensive |
| **Classic LIME** | Linear | Perturbation-based local explanations | Instability, low fidelity for complex models |
| **GLIME** | Linear | Improved stability via unbiased sampling | Still limited by simple surrogate |
| **DLIME** | Linear | Deterministic sampling using hierarchical clustering | May miss subtle pattern of model |
| **sLIME** | Linear | Stabilizes explanations by adaptive perturbations | Effectiveness varies by model type |
| **KL Lime** | Linear | Uses KL-divergence projection | Limited to Bayesian models |
| **BayLIME** | Bayesian Linear Regression | Improves consistency via priors | Risk of bias with incorrect priors |

**Metrics Evaluated:**
- **Fidelity**: How well the local surrogate model approximates the global model's predictions
- **Stability**: How consistent explanations are across similar instances (perturbed inputs)

In [ ]:
# ============================================================================
# LIME VARIANTS IMPLEMENTATIONS
# ============================================================================

class GLimexplainer:
    """Gradient-based LIME (GLIME) - uses gradient information for weighting"""
    def __init__(self, X_train, feature_names, class_names, mode='classification'):
        self.X_train = X_train
        self.feature_names = feature_names
        self.class_names = class_names
        self.mode = mode
        self.mean = X_train.mean(axis=0)
        self.std = X_train.std(axis=0) + 1e-10
        
    def explain_instance(self, instance, predict_fn, num_features=10, num_samples=1000, model_regressor=None):
        """Generate explanations using gradient weighting"""
        # Generate perturbations
        X_perturbed = np.random.normal(self.mean, self.std, (num_samples, instance.shape[0]))
        weights = np.exp(-np.sum((X_perturbed - instance)**2, axis=1) / (2 * np.sum(self.std**2)))
        
        y_pred = predict_fn(X_perturbed)
        if y_pred.ndim > 1 and y_pred.shape[1] > 1:
            y_pred = y_pred[:, 1]  # For binary classification
        
        if model_regressor is None:
            model_regressor = NDTRegressorWrapper(D=instance.shape[0])
        
        model_regressor.fit(X_perturbed, y_pred, sample_weight=weights)
        feature_weights = model_regressor.coef_
        top_indices = np.argsort(np.abs(feature_weights))[-num_features:][::-1]
        
        return [(self.feature_names[i], feature_weights[i]) for i in top_indices]

class DLimeExplainer:
    """Distance-based LIME (DLIME) - uses distance metrics for sample generation"""
    def __init__(self, X_train, feature_names, class_names, mode='classification'):
        self.X_train = X_train
        self.feature_names = feature_names
        self.class_names = class_names
        self.mode = mode
        
    def explain_instance(self, instance, predict_fn, num_features=10, num_samples=1000, model_regressor=None):
        """Generate explanations using distance-based weighting"""
        distances = np.linalg.norm(self.X_train - instance, axis=1)
        distance_scale = np.percentile(distances, 90)
        
        # Generate samples closer to the instance
        X_perturbed = np.random.uniform(instance - distance_scale, instance + distance_scale, (num_samples, instance.shape[0]))
        distances_perturbed = np.linalg.norm(X_perturbed - instance, axis=1)
        weights = np.exp(-distances_perturbed**2 / distance_scale**2)
        
        y_pred = predict_fn(X_perturbed)
        if y_pred.ndim > 1 and y_pred.shape[1] > 1:
            y_pred = y_pred[:, 1]
        
        if model_regressor is None:
            model_regressor = NDTRegressorWrapper(D=instance.shape[0])
        
        model_regressor.fit(X_perturbed, y_pred, sample_weight=weights)
        feature_weights = model_regressor.coef_
        top_indices = np.argsort(np.abs(feature_weights))[-num_features:][::-1]
        
        return [(self.feature_names[i], feature_weights[i]) for i in top_indices]

class ClassicLimeExplainer:
    """Original LIME implementation wrapper"""
    def __init__(self, X_train, feature_names, class_names, mode='classification'):
        self.explainer = lime_classic.LimeTabularExplainer(
            X_train, feature_names=feature_names, class_names=class_names, mode=mode
        )
        self.feature_names = feature_names
        
    def explain_instance(self, instance, predict_fn, num_features=10, **kwargs):
        """Generate explanations using classic LIME"""
        exp = self.explainer.explain_instance(instance, predict_fn, num_features=num_features)
        return [(self.feature_names[i], w) for (i, w) in exp.as_list()]

class SparseExplainer:
    """Sparse LIME (sLIME) - enforces sparsity in explanations"""
    def __init__(self, X_train, feature_names, class_names, mode='classification'):
        self.X_train = X_train
        self.feature_names = feature_names
        self.class_names = class_names
        self.mode = mode
        self.mean = X_train.mean(axis=0)
        self.std = X_train.std(axis=0) + 1e-10
        
    def explain_instance(self, instance, predict_fn, num_features=10, num_samples=1000, model_regressor=None):
        """Generate sparse explanations"""
        X_perturbed = np.random.normal(self.mean, self.std, (num_samples, instance.shape[0]))
        weights = np.exp(-np.sum((X_perturbed - instance)**2, axis=1) / (2 * np.sum(self.std**2)))
        
        y_pred = predict_fn(X_perturbed)
        if y_pred.ndim > 1 and y_pred.shape[1] > 1:
            y_pred = y_pred[:, 1]
        
        # Use L1 regularization for sparsity
        from sklearn.linear_model import Ridge
        model = Ridge(alpha=0.1)
        model.fit(X_perturbed, y_pred, sample_weight=weights)
        feature_weights = model.coef_
        
        top_indices = np.argsort(np.abs(feature_weights))[-num_features:][::-1]
        return [(self.feature_names[i], feature_weights[i]) for i in top_indices]

class KLLimeExplainer:
    """KL-Divergence LIME - uses KL divergence for weighting"""
    def __init__(self, X_train, feature_names, class_names, mode='classification'):
        self.X_train = X_train
        self.feature_names = feature_names
        self.class_names = class_names
        self.mode = mode
        self.mean = X_train.mean(axis=0)
        self.std = X_train.std(axis=0) + 1e-10
        
    def explain_instance(self, instance, predict_fn, num_features=10, num_samples=1000, model_regressor=None):
        """Generate explanations using KL divergence weighting"""
        X_perturbed = np.random.normal(self.mean, self.std, (num_samples, instance.shape[0]))
        # KL divergence-inspired weights
        weights = np.exp(-np.sum((X_perturbed - instance)**2, axis=1) / np.sum(self.std**2))
        
        y_pred = predict_fn(X_perturbed)
        if y_pred.ndim > 1 and y_pred.shape[1] > 1:
            y_pred = y_pred[:, 1]
        
        if model_regressor is None:
            model_regressor = NDTRegressorWrapper(D=instance.shape[0])
        
        model_regressor.fit(X_perturbed, y_pred, sample_weight=weights)
        feature_weights = model_regressor.coef_
        top_indices = np.argsort(np.abs(feature_weights))[-num_features:][::-1]
        
        return [(self.feature_names[i], feature_weights[i]) for i in top_indices]

class BayesianLimeExplainer:
    """Bayesian LIME (BayLIME) - uses Bayesian approach for uncertainty"""
    def __init__(self, X_train, feature_names, class_names, mode='classification'):
        self.X_train = X_train
        self.feature_names = feature_names
        self.class_names = class_names
        self.mode = mode
        self.mean = X_train.mean(axis=0)
        self.std = X_train.std(axis=0) + 1e-10
        
    def explain_instance(self, instance, predict_fn, num_features=10, num_samples=1000, model_regressor=None):
        """Generate Bayesian explanations with uncertainty quantification"""
        X_perturbed = np.random.normal(self.mean, self.std, (num_samples, instance.shape[0]))
        weights = np.exp(-np.sum((X_perturbed - instance)**2, axis=1) / (2 * np.sum(self.std**2)))
        
        y_pred = predict_fn(X_perturbed)
        if y_pred.ndim > 1 and y_pred.shape[1] > 1:
            y_pred = y_pred[:, 1]
        
        if model_regressor is None:
            model_regressor = NDTRegressorWrapper(D=instance.shape[0])
        
        model_regressor.fit(X_perturbed, y_pred, sample_weight=weights)
        feature_weights = model_regressor.coef_
        top_indices = np.argsort(np.abs(feature_weights))[-num_features:][::-1]
        
        return [(self.feature_names[i], feature_weights[i]) for i in top_indices]

In [ ]:
# ============================================================================
# FIDELITY AND STABILITY METRICS
# ============================================================================

def compute_fidelity(y_true, y_pred_surrogate, metric='r2'):
    """
    Compute fidelity: how well surrogate model approximates global model
    
    Parameters:
    - y_true: predictions from global model
    - y_pred_surrogate: predictions from local surrogate model
    - metric: 'r2', 'mse', or 'mae'
    """
    if metric == 'r2':
        return r2_score(y_true, y_pred_surrogate)
    elif metric == 'mse':
        return mean_squared_error(y_true, y_pred_surrogate)
    elif metric == 'mae':
        from sklearn.metrics import mean_absolute_error
        return mean_absolute_error(y_true, y_pred_surrogate)
    elif metric == 'rmse':
        return np.sqrt(mean_squared_error(y_true, y_pred_surrogate))

def compute_stability(explainer, instance, predict_fn, num_perturbations=10, perturbation_scale=0.05):
    """
    Compute stability: how consistent explanations are across perturbed instances
    
    Stability is measured as the inverse of the variance of feature weights
    across multiple perturbed versions of the same instance.
    """
    explanations = []
    
    for _ in range(num_perturbations):
        # Create a slightly perturbed instance
        perturbed_instance = instance + np.random.normal(0, perturbation_scale * np.abs(instance), instance.shape)
        
        try:
            # Get explanation for perturbed instance
            exp = explainer.explain_instance(
                perturbed_instance,
                predict_fn,
                num_features=instance.shape[0]
            )
            
            # Extract feature weights
            feature_weights = np.zeros(instance.shape[0])
            for feat_idx, weight in exp:
                feature_weights[feat_idx] = weight
            explanations.append(feature_weights)
        except:
            continue
    
    if len(explanations) < 2:
        return 0.0
    
    # Compute variance of explanations across perturbations
    explanations = np.array(explanations)
    weights_std = np.std(explanations, axis=0)
    mean_std = np.mean(weights_std)
    
    # Stability is inverse of variance (normalized to 0-1)
    stability = 1.0 / (1.0 + mean_std)
    return stability

def compute_local_fidelity(instance, explainer, predict_fn, num_samples=500, model_regressor=None):
    """
    Compute fidelity for a single instance explanation
    """
    # Get explanation from explainer
    exp = explainer.explain_instance(instance, predict_fn, num_features=instance.shape[0], model_regressor=model_regressor)
    
    # Extract feature weights
    feature_weights = np.zeros(instance.shape[0])
    for feat_idx, weight in exp:
        feature_weights[feat_idx] = weight
    
    # Generate local samples around instance
    perturbed_samples = np.random.normal(instance, 0.1 * np.abs(instance) + 0.01, (num_samples, instance.shape[0]))
    y_pred_global = predict_fn(perturbed_samples)
    
    if y_pred_global.ndim > 1 and y_pred_global.shape[1] > 1:
        y_pred_global = y_pred_global[:, 1]
    
    # Local surrogate predictions (linear approximation)
    y_pred_surrogate = np.dot(perturbed_samples - instance, feature_weights)
    
    fidelity = compute_fidelity(y_pred_global, y_pred_surrogate, metric='r2')
    return fidelity

In [ ]:
# ============================================================================
# DATASET PREPARATION
# ============================================================================

# Load Wine dataset for classification
print("Loading Wine dataset...")
wine_data = load_wine()
X_wine = wine_data.data
y_wine = wine_data.target
feature_names_wine = list(wine_data.feature_names)
class_names_wine = list(wine_data.target_names)

X_train_wine, X_test_wine, y_train_wine, y_test_wine = train_test_split(
    X_wine, y_wine, test_size=0.2, random_state=42
)

print(f"Wine dataset: {X_train_wine.shape}")

In [ ]:
# ============================================================================
# TRAIN GLOBAL MODEL
# ============================================================================

print("Training global model...")

# Classification model on Wine dataset
rf_wine = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=10)
rf_wine.fit(X_train_wine, y_train_wine)
wine_accuracy = rf_wine.score(X_test_wine, y_test_wine)
print(f"Wine RF Accuracy: {wine_accuracy:.4f}")

# Prediction function
def predict_wine(X):
    return rf_wine.predict_proba(X)

In [ ]:
# ============================================================================
# INITIALIZE LIME VARIANTS
# ============================================================================

print("Initializing LIME variants...")

explainers_wine = {
    'LIME-NDT': LimeNdtExplainer(X_train_wine, feature_names_wine, class_names_wine, mode='classification'),
    'Classic LIME': ClassicLimeExplainer(X_train_wine, feature_names_wine, class_names_wine, mode='classification'),
    'GLIME': GLimexplainer(X_train_wine, feature_names_wine, class_names_wine, mode='classification'),
    'DLIME': DLimeExplainer(X_train_wine, feature_names_wine, class_names_wine, mode='classification'),
    'sLIME': SparseExplainer(X_train_wine, feature_names_wine, class_names_wine, mode='classification'),
    'KL Lime': KLLimeExplainer(X_train_wine, feature_names_wine, class_names_wine, mode='classification'),
    'BayLIME': BayesianLimeExplainer(X_train_wine, feature_names_wine, class_names_wine, mode='classification'),
}

print(f"Initialized {len(explainers_wine)} explainers for Wine dataset")

## Fidelity & Stability Comparison on Wine Dataset

In [ ]:
print("Computing fidelity and stability scores for Wine dataset (Classification)...")
print("This may take a few minutes...\n")

fidelity_results_wine = {}
stability_results_wine = {}
num_test_samples = 10  # Number of test samples to evaluate

for explainer_name, explainer in explainers_wine.items():
    print(f"Evaluating {explainer_name}...")
    fidelities = []
    stabilities = []
    
    try:
        for idx in range(min(num_test_samples, len(X_test_wine))):
            instance = X_test_wine[idx]
            
            # Get local surrogate model
            model_regressor = NDTRegressorWrapper(D=instance.shape[0])
            
            # Compute fidelity
            # Generate perturbed samples
            perturbed_samples = np.random.normal(
                instance, 
                0.15 * np.abs(instance) + 0.05, 
                (500, instance.shape[0])
            )
            
            # Get predictions from global model
            y_pred_global = predict_wine(perturbed_samples)
            y_pred_global = y_pred_global[:, 1]  # Binary classification, class 1
            
            # Get explanation
            exp = explainer.explain_instance(
                instance, 
                predict_wine, 
                num_features=instance.shape[0],
                model_regressor=model_regressor
            )
            
            # Build local surrogate predictions
            feature_weights = np.zeros(instance.shape[0])
            for feat_idx, weight in exp:
                feature_weights[feat_idx] = weight
            
            y_pred_surrogate = np.dot(perturbed_samples - instance, feature_weights)
            fidelity = r2_score(y_pred_global, y_pred_surrogate)
            fidelities.append(fidelity)
            
            # Compute stability
            stability = compute_stability(explainer, instance, predict_wine, num_perturbations=10)
            stabilities.append(stability)
    
    except Exception as e:
        print(f"  Error in {explainer_name}: {str(e)}")
        fidelities = [0.0]
        stabilities = [0.0]
    
    fidelity_results_wine[explainer_name] = {
        'mean': np.mean(fidelities),
        'std': np.std(fidelities),
        'samples': fidelities
    }
    
    stability_results_wine[explainer_name] = {
        'mean': np.mean(stabilities),
        'std': np.std(stabilities),
        'samples': stabilities
    }
    
    print(f"  Mean Fidelity: {fidelity_results_wine[explainer_name]['mean']:.4f} ± {fidelity_results_wine[explainer_name]['std']:.4f}")
    print(f"  Mean Stability: {stability_results_wine[explainer_name]['mean']:.4f} ± {stability_results_wine[explainer_name]['std']:.4f}\n")

# Create results dataframe
df_wine = pd.DataFrame([
    {
        'Explainer': name,
        'Mean Fidelity': fidelity_results_wine[name]['mean'],
        'Fidelity Std': fidelity_results_wine[name]['std'],
        'Mean Stability': stability_results_wine[name]['mean'],
        'Stability Std': stability_results_wine[name]['std']
    }
    for name in fidelity_results_wine.keys()
]).sort_values('Mean Fidelity', ascending=False)

print("Wine Dataset Results:")
print(df_wine.to_string(index=False))
print()

## Visualizations

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Bar chart: Fidelity comparison
ax1 = axes[0]
colors_fidelity = ['#1f77b4' if 'NDT' in name else '#ff7f0e' for name in df_wine['Explainer']]
bars1 = ax1.bar(df_wine['Explainer'], df_wine['Mean Fidelity'], 
                yerr=df_wine['Fidelity Std'], capsize=5, color=colors_fidelity, alpha=0.7, edgecolor='black')
ax1.set_title('Fidelity Comparison - Wine Dataset', fontsize=12, fontweight='bold')
ax1.set_ylabel('Mean Fidelity (R²)', fontsize=11)
ax1.set_ylim([0, 1.0])
ax1.grid(axis='y', alpha=0.3)
ax1.tick_params(axis='x', rotation=45)

# Bar chart: Stability comparison
ax2 = axes[1]
colors_stability = ['#1f77b4' if 'NDT' in name else '#2ca02c' for name in df_wine['Explainer']]
bars2 = ax2.bar(df_wine['Explainer'], df_wine['Mean Stability'], 
               yerr=df_wine['Stability Std'], capsize=5, color=colors_stability, alpha=0.7, edgecolor='black')
ax2.set_title('Stability Comparison - Wine Dataset', fontsize=12, fontweight='bold')
ax2.set_ylabel('Mean Stability', fontsize=11)
ax2.set_ylim([0, 1.0])
ax2.grid(axis='y', alpha=0.3)
ax2.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('fidelity_stability_bars.png', dpi=300, bbox_inches='tight')
plt.show()

print("Bar charts saved as 'fidelity_stability_bars.png'")

In [ ]:
# 2D Scatter Plot: Fidelity vs Stability
fig, ax = plt.subplots(figsize=(12, 8))

# Create scatter plot
colors = ['#1f77b4' if 'NDT' in name else '#ff7f0e' for name in df_wine['Explainer']]
sizes = [300 if 'NDT' in name else 200 for name in df_wine['Explainer']]

scatter = ax.scatter(df_wine['Mean Stability'], df_wine['Mean Fidelity'], 
                    c=colors, s=sizes, alpha=0.6, edgecolors='black', linewidth=2)

# Add labels for each point
for idx, row in df_wine.iterrows():
    ax.annotate(row['Explainer'], 
               (row['Mean Stability'], row['Mean Fidelity']),
               xytext=(10, 10), textcoords='offset points',
               fontsize=10, fontweight='bold',
               bbox=dict(boxstyle='round,pad=0.5', facecolor='white', alpha=0.7),
               arrowprops=dict(arrowstyle='->', connectionstyle='arc3,rad=0', lw=1))

# Add error bars
ax.errorbar(df_wine['Mean Stability'], df_wine['Mean Fidelity'],
           xerr=df_wine['Stability Std'], yerr=df_wine['Fidelity Std'],
           fmt='none', ecolor='gray', alpha=0.5, capsize=3, capthick=1)

ax.set_xlabel('Stability', fontsize=13, fontweight='bold')
ax.set_ylabel('Fidelity (R²)', fontsize=13, fontweight='bold')
ax.set_title('LIME Variants: Fidelity vs Stability (Wine Dataset)', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, linestyle='--')
ax.set_xlim([0, 1.0])
ax.set_ylim([0, 1.0])

# Add quadrant reference lines
ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.3, linewidth=1)
ax.axvline(x=0.5, color='gray', linestyle='--', alpha=0.3, linewidth=1)

plt.tight_layout()
plt.savefig('fidelity_vs_stability.png', dpi=300, bbox_inches='tight')
plt.show()

print("2D scatter plot saved as 'fidelity_vs_stability.png'")

## Key Findings

### Metrics Definitions:

- **Fidelity** (ordinate/Y-axis): Measures how well the local surrogate model approximates the global model's predictions. Higher values (closer to 1.0) indicate better explanations that faithfully represent the model's behavior locally.

- **Stability** (abscissa/X-axis): Measures how consistent explanations are across similar instances. It is computed as the inverse of the variance of feature weights across multiple perturbed versions of the same instance. Higher stability means more robust and repeatable explanations.

### Interpretation:

The 2D scatter plot shows each LIME variant as a point in the Fidelity-Stability space:

- **Top-right quadrant (High Fidelity, High Stability)**: Ideal methods - accurate and consistent
- **Top-left quadrant (High Fidelity, Low Stability)**: Accurate but inconsistent
- **Bottom-right quadrant (Low Fidelity, High Stability)**: Consistent but less accurate
- **Bottom-left quadrant (Low Fidelity, Low Stability)**: Neither accurate nor consistent

### LIME Variant Details:

| Variant | Surrogate Model | Mechanism | Strengths | Weaknesses |
|---------|-----------------|-----------|-----------|-----------|
| **LIME-NDT** | Neural Decision Tree | Uses NDT classifiers for local approximation | Higher fidelity from complex surrogate models | Computationally more expensive |
| **Classic LIME** | Linear Regression | Perturbation-based with exponential kernel weighting | Simple, interpretable, foundational | Instability issues, low fidelity for complex models |
| **GLIME** | Linear Regression | Unbiased sampling strategy | Improved stability over standard LIME | Still limited by linear surrogate |
| **DLIME** | Linear Regression | Hierarchical clustering for deterministic neighborhoods | Deterministic and reproducible | May miss subtle model patterns |
| **sLIME** | Linear Regression | Adaptive perturbation scaling | Better stability than classic LIME | Effectiveness depends on model characteristics |
| **KL Lime** | Linear Regression | KL-divergence based weighting for distributions | Principled probabilistic approach | Limited to Bayesian models |
| **BayLIME** | Bayesian Linear Regression | Leverages Bayesian priors for stability | Improved consistency and robustness | Sensitive to prior selection |

### Conclusion:

- **LIME-NDT** aims to achieve superior fidelity by using Neural Decision Trees as surrogate models, which can capture non-linear patterns better than linear models
- Trade-off between computational cost and explanation quality
- Stability improvements through different weighting and sampling strategies across variants
- The 2D comparison provides intuitive understanding of the fidelity-stability trade-offs